In [ ]:
# 패키지 설치 (Colab/로컬 공통)
%pip install -q langchain langchain-openai langchain-core python-dotenv gradio

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-4o-mini"

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("프로젝트 .env에 OPENAI_API_KEY를 설정하세요.")

print(f"API 키 로드 완료 / 모델: {MODEL_NAME}")


In [ ]:
from langchain_core.runnables import RunnableLambda

def add_one(x):
    return x + 1

r_add = RunnableLambda(add_one)
print(r_add.invoke(3))      # 4
print(r_add.batch([1, 2, 3]))  # [2, 3, 4]

In [ ]:
from langchain_core.runnables import RunnableLambda

r_add = RunnableLambda(lambda x: x + 1)
r_double = RunnableLambda(lambda x: x * 2)
4
chain_a = r_add | r_double 
chain_b = r_double | r_add

print(chain_a.invoke(3))  # 8
print(chain_b.invoke(3))  # 7
print(chain_a.batch([1, 2, 3]))  # [4, 6, 8]
print(chain_b.batch([1, 2, 3]))  # [3, 5, 7]
print(type(chain_a))  # 'chain'

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

analyer = RunnableParallel(
    original=RunnablePassthrough(),
    upper = RunnableLambda(lambda x: x.upper()),
    length = RunnableLambda(lambda x: len(x))
)

print(analyer.invoke("hello"))  # {'original': 'hello', 'upper': 'HELLO', 'length': 5}

| 단계 | 들어오는 것 | 나가는 것 |
| --- | --- | --- |
| `prompt` | `{"topic": "인공지능"}` (딕셔너리) | 완성된 프롬프트 메시지 |
| `model` | 프롬프트 메시지 | `AIMessage` (답변 + 부가정보) |
| `parser` | `AIMessage` | 문자열 |

### 실습.  LLM 체인 만들기 (API 키 필요)

In [33]:
prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 초보자도 이해할 수 있도록 설명해 주세요."
)
model = ChatOpenAI(model=MODEL_NAME)
parser = StrOutputParser()

chain = prompt | model | parser

print(chain.invoke({"topic": "인공지능"}))

인공지능(Artificial Intelligence, AI)은 컴퓨터가 사람처럼 생각하고 학습하며 문제를 해결할 수 있게 하는 기술을 말합니다. 쉽게 말해, 컴퓨터에게 "스스로 뭔가를 할 수 있게 만드는 방법"이라고 할 수 있습니다.

### 인공지능의 주요 개념

1. **기계 학습(Machine Learning)**: AI의 한 분야로, 컴퓨터가 데이터에서 패턴을 배우고 스스로 개선할 수 있게 만드는 기술입니다. 예를 들어, 많은 이미지 데이터를 통해 "고양이"와 "개"를 구분하는 방법을 배우는 것입니다.

2. **딥 러닝(Deep Learning)**: 기계 학습의 한 부분으로, 신경망(neural networks)이라는 구조를 사용해 더 복잡한 데이터, 예를 들어 음성 인식이나 이미지 인식을 할 수 있게 합니다. 마치 사람의 뇌처럼 여러 층의 연결로 이루어져 있습니다.

3. **자연어 처리(Natural Language Processing, NLP)**: 이 기술은 컴퓨터가 인간의 언어를 이해하고 해석하는 데 사용됩니다. 예를 들어, 챗봇이 고객 질문에 답변하거나 문장을 번역하는 등의 작업을 합니다.

### 인공지능의 활용 예시

- **스마트폰 비서**: Siri나 Google Assistant처럼 음성으로 명령을 듣고 답변하는 시스템.
- **추천 시스템**: Netflix나 YouTube가 개인의 시청 기록을 바탕으로 영화를 추천하는 기능.
- **자율주행차**: AI가 센서를 사용해 주변 환경을 인식하고, 안전하게 주행하는 기술.

### 요약

인공지능은 컴퓨터가 인간처럼 생각하도록 만드는 기술입니다. 이를 통해 여러 가지 작업을 더 효율적으로 수행할 수 있고, 일상생활에서 많은 도움을 받을 수 있습니다. 인공지능은 여전히 발전 중이며, 우리의 미래에 큰 영향을 미칠 기술입니다.


### 여러 주제 batch 로 병렬

In [ ]:
results = chain.batch([
    {"topic": "인공지능"},
    {"topic": "머신러닝"}
])

#for r in results:
#    print(f"\n'{r}'")  # 각 결과 출력

for chunk in chain.stream({"topic": "모델 양자화"}):
    print(chunk, end="", flush=True)

모델 양자화(Model Quantization)는 머신 러닝과 딥 러닝에서 사용되는 기술 중 하나로, 모델의 크기와 연산 효율성을 최적화하기 위해 파라미터와 계산을 적은 비트수로 표현하는 과정을 말합니다. 이 과정을 통해 모델을 더 빠르게 실행하고, 메모리 사용량을 줄이며, 전력 소모를 줄일 수 있습니다. 아래에 좀 더 쉽게 설명해볼게요.

### 모델 양자화의 기본 개념

1. **떠 오른 숫자들 (Float vs. Integer)**:
   - 통상 딥 러닝 모델은 실수(Float) 형태로 파라미터를 저장합니다. 예를 들어, 32비트 부동소수점 숫자가 일반적입니다.
   - 양자화는 이 숫자들을 정수(Integer)로 변환하여 저장합니다. 예를 들어, 8비트 정수로 변환할 수 있습니다. 

2. **왜 양자화할까요?**:
   - **메모리 절약**: 32비트를 8비트로 줄이면 저장 공간이 크게 줄어듭니다. 예를 들어, 1GB 용량의 모델이 4배 작아질 수 있습니다.
   - **속도 개선**: 정수 연산은 부동소수점 연산보다 더 빠르게 처리할 수 있습니다. 특히 모바일 기기나 경량 장치에서 유리합니다.
   - **전력 효율**: 양자화된 모델은 더 적은 전력을 소모하므로, 배터리로 구동되는 기기에서 특히 유용합니다.

### 양자화의 방법

1. **가중치 양자화 (Weight Quantization)**:
   - 가중치를 정수로 변환하는 과정입니다. 예를 들어, 모델의 각 파라미터를 0~255의 범위로 표현할 수 있습니다.

2. **활성화 양자화 (Activation Quantization)**:
   - 네트워크의 출력 값도 양자화하여 정수로 변환합니다. 이렇게 하면 중간 결과도 메모리와 연산에서 이점을 얻을 수 있습니다.

3. **훈련 후 양자화 (Post-Training Quantization)**:
   - 모델을 훈련한 후에 양자화를 진행하는 방법입니다. 이 경우 기존의 학습된 모델을 그대로 사용하면서, 필요한 부문만 양자화합니다.

#

**과제 1 (기초, API 키 불필요)** — 문자열을 받아 양쪽 공백을 없애고, 대문자로 바꾼 뒤, 끝에 `!`를 붙이는 체인을 만드세요.

```python
# 입력: "  hello world  "
# 출력: "HELLO WORLD!"
```

In [ ]:
from langchain_core.runnables import RunnableLambda

original = "  Hello, World   "
strip = RunnableLambda(lambda x: x.strip())
pector = RunnableLambda(lambda x: x + "!")

chain = strip | pector

print(chain.invoke(original))  # "Hello, World!"

Hello, World!


**과제 2 (중급, API 키 불필요)** — 숫자 리스트를 받아 합계, 평균, 최댓값을 한 번에 딕셔너리로 돌려주는 Runnable을 만드세요.

```python
# 입력: [10, 20, 30]
# 출력: {'total': 60, 'average': 20.0, 'max': 30}
```

In [53]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

# total = sum(n)
# aver = sum(n) / len(n)
# maximum = max(n)

# RunnableLambda를 통해 각 계산을 수행하는 람다 함수를 정의
totalnum = RunnableLambda(lambda x: sum(x))
aver = RunnableLambda(lambda x: sum(x) / len(x))
maximum = RunnableLambda(lambda x: max(x))

# RunnableParallel의 인자를 딕셔너리키를 통해 이름을 지정
chain =  RunnableParallel(
    total=totalnum,
    average=aver,
    max=maximum,
)

print(chain.invoke([10, 20, 30]))

{'total': 60, 'average': 20.0, 'max': 30}


**과제 3 (심화, API 키 필요)** — 주제 하나를 넣으면 ① 한 줄 설명과 ② 관련 키워드 3개를 동시에 만들어 딕셔너리로 돌려주는 체인을 만드세요.

```python
# 입력: {"topic": "머신러닝"}
# 출력: {'summary': '...', 'keywords': '...'}
```

In [57]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI

#1 프롬프트 정의
summary = ChatPromptTemplate.from_template(
    "{topic}을 초보자도 이해할 수 있도록 한 줄로 설명"
)
keywords = ChatPromptTemplate.from_template(
    "{topic}과 관련된 키워드 3개만 쉼표로 구분해 출력"
)

# LLM 정의
llm = ChatOpenAI(model=MODEL_NAME)

# chain 구성
# RunnableParallel를 통해 여러 개의 Runnable을 병렬로 실행
chain = RunnableParallel(
# summary 와 keywords를 각각 실행하고 결과를 딕셔너리로 반환
    summary=summary | llm | StrOutputParser(),
    keywords=keywords | llm | StrOutputParser(),
)

print(chain.invoke({"topic": "머신러닝"}))

{'summary': '머신러닝은 컴퓨터가 데이터에서 패턴을 배우고, 이를 통해 예측이나 결정을 자동으로 수행하도록 하는 기술입니다.', 'keywords': '지도학습, 비지도학습, 신경망'}
